# LeetCode 2182 — Construct String With Repeat Limit bằng Python

Link đề: https://leetcode.com/problems/construct-string-with-repeat-limit/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Greedy, Heap (Priority Queue), Counting

Skeleton LeetCode:

```python
class Solution:
    def repeatLimitedString(self, s: str, repeatLimit: int) -> str:
        ...
```

Dựng xâu **lớn nhất theo thứ tự từ điển** từ các ký tự của `s` (không bắt buộc dùng hết) sao cho không ký tự nào lặp quá `repeatLimit` lần liên tiếp. Đây là bài **tham lam theo từng vị trí**: ở mỗi bước đặt ký tự lớn nhất **được phép**. Notebook đi từ **vét cạn mọi xâu dựng được** → **tham lam từng ký tự với mảng đếm 26 ô** → **tham lam theo khối** (đặt `min(cnt, limit)` ký tự một lần, chèn một ký tự nhỏ hơn) → **max-heap** → **hai con trỏ trên mảng đếm `O(n + 26)`**.

## 1. Mục tiêu học

1. Hiểu so sánh **thứ tự từ điển** khi độ dài khác nhau: xâu dài hơn lớn hơn nếu xâu ngắn là tiền tố.
2. Chứng minh tham lam "**mỗi vị trí đặt ký tự lớn nhất còn hợp lệ**" là tối ưu.
3. Cài đặt theo **khối**: đặt tối đa `repeatLimit` bản của ký tự lớn nhất, rồi **chỉ một** ký tự nhỏ hơn kế tiếp để "ngắt", lặp lại.
4. Dùng **max-heap** (đẩy `-ord`) hoặc **hai con trỏ** trên mảng đếm.
5. Kiểm thử bằng vét cạn trên xâu ngắn + validator (giới hạn lặp, đa tập ký tự con).

## 2. Đề bài & ví dụ

Cho xâu `s` và số nguyên `repeatLimit`. Dựng xâu mới `repeatLimitedString` bằng các ký tự của `s` sao cho **không** chữ cái nào xuất hiện **quá `repeatLimit` lần liên tiếp**. Không cần dùng hết ký tự. Trả về xâu như vậy **lớn nhất theo thứ tự từ điển**.

(Thứ tự từ điển: ở vị trí đầu tiên khác nhau, xâu có chữ cái đứng sau trong bảng chữ cái lớn hơn; nếu `min(len)` ký tự đầu giống nhau thì xâu **dài hơn** lớn hơn.)

**Ví dụ 1:** `s = "cczazcc"`, `repeatLimit = 3` → `"zzcccac"`.

**Ví dụ 2:** `s = "aababab"`, `repeatLimit = 2` → `"bbabaa"` (không dùng hết `a`).

**Ràng buộc:** `1 ≤ repeatLimit ≤ len(s) ≤ 10⁵`, `s` gồm chữ cái thường.

In [1]:
import heapq
import random
import time
from collections import Counter
from itertools import permutations
from typing import List

examples = [("cczazcc", 3, "zzcccac"), ("aababab", 2, "bbabaa")]

def max_run(t):
    best = cur = 0
    for i, ch in enumerate(t):
        cur = cur + 1 if i and t[i - 1] == ch else 1
        best = max(best, cur)
    return best

for s, k, exp in examples:
    print(f"s={s!r}, limit={k}: đếm {dict(sorted(Counter(s).items(), reverse=True))}, kỳ vọng {exp!r} (run dài nhất {max_run(exp)})")

s='cczazcc', limit=3: đếm {'z': 2, 'c': 4, 'a': 1}, kỳ vọng 'zzcccac' (run dài nhất 3)
s='aababab', limit=2: đếm {'b': 3, 'a': 4}, kỳ vọng 'bbabaa' (run dài nhất 2)


## 3. Kiến thức nền

### 3.1 So sánh xâu trong Python = thứ tự từ điển

In [2]:
print('"zzc" > "zz"?', "zzc" > "zz", '| "zzcccac" > "zzccca"?', "zzcccac" > "zzccca", '| "b" > "abbbb"?', "b" > "abbbb")

"zzc" > "zz"? True | "zzcccac" > "zzccca"? True | "b" > "abbbb"? True


### 3.2 Mảng đếm 26 ô

In [3]:
cnt = [0] * 26
for ch in "cczazcc":
    cnt[ord(ch) - 97] += 1
print({chr(97 + i): c for i, c in enumerate(cnt) if c})

{'a': 1, 'c': 4, 'z': 2}


### 3.3 Max-heap bằng `heapq` (đẩy giá trị âm)

In [4]:
h = []
for ch, c in Counter("cczazcc").items():
    heapq.heappush(h, (-ord(ch), c))
print("ký tự lớn nhất:", chr(-h[0][0]), "| thứ tự lấy ra:", [chr(-heapq.heappop(h)[0]) for _ in range(len(h))])

ký tự lớn nhất: z | thứ tự lấy ra: ['z', 'c', 'a']


### 3.4 Validator cho kết quả

Một xâu `t` hợp lệ ⇔ (1) mỗi ký tự dùng không quá số lần có trong `s` (đa tập con), (2) không có run dài hơn `repeatLimit`.

In [5]:
def is_valid(s, k, t):
    cs, ct = Counter(s), Counter(t)
    return all(ct[c] <= cs[c] for c in ct) and max_run(t) <= k

print(is_valid("cczazcc", 3, "zzcccac"), is_valid("cczazcc", 3, "zzcccca"), is_valid("aababab", 2, "bbabaaa"))

True False False


## 4. Ý tưởng nền tảng / trực giác

Muốn xâu lớn nhất theo từ điển ⇒ **vị trí đầu** phải là ký tự lớn nhất có thể, rồi vị trí thứ hai lớn nhất có thể, … Ràng buộc duy nhất: không đặt ký tự `c` nếu đã có `repeatLimit` bản `c` liên tiếp ngay trước.

Tính tay ví dụ 1 `s = "cczazcc"` (đếm `z:2, c:4, a:1`), `limit = 3`:
1. Đặt `z, z` (hết `z`) ⇒ `zz`.
2. Ký tự lớn nhất còn lại `c`: đặt 3 bản ⇒ `zzccc` (đã chạm giới hạn, còn 1 `c`).
3. Không được đặt `c` nữa ⇒ đặt ký tự lớn nhất **khác** `c`: `a` (một bản) ⇒ `zzccca`.
4. Giờ lại được đặt `c` ⇒ `zzcccac`. Hết ký tự.

Ví dụ 2 `"aababab"` (`b:3, a:4`), `limit = 2`: `bb` → phải ngắt bằng `a` → `bba` → `b` → `bbab` → không còn `b`, đặt `aa` (tới giới hạn) → `bbabaa`; còn 1 `a` nhưng không có gì để ngắt ⇒ dừng.

**Điểm mấu chốt:** khi "ngắt", chỉ đặt **một** ký tự nhỏ hơn — đặt nhiều hơn sẽ làm vị trí kế tiếp nhỏ đi (vì ngay sau đó ta lại được đặt ký tự lớn).

## 5. Cách 1 — Brute Force / baseline: thử mọi xâu dựng được

Sinh mọi xâu từ đa tập con của `s` (mọi độ dài), lọc hợp lệ, lấy `max`. Chỉ khả thi với `len(s) ≤ 7`.

In [6]:
def rls_brute(s, k):
    best = ""
    chars = sorted(s)
    seen = set()
    n = len(chars)
    for L in range(1, n + 1):
        for p in permutations(chars, L):
            if p in seen:
                continue
            seen.add(p)
            t = "".join(p)
            if t > best and max_run(t) <= k:
                best = t
    return best

for s, k, exp in examples:
    print(repr(rls_brute(s, k)), "(kỳ vọng", repr(exp), ")")

'zzcccac' (kỳ vọng 'zzcccac' )
'bbabaa' (kỳ vọng 'bbabaa' )


**Độ phức tạp:** `Σ_L n!/(n−L)!` hoán vị — bùng nổ. **Nút thắt:** cấu trúc nghiệm đã rõ (tham lam theo vị trí) ⇒ không cần thử.

## 6. Ý tưởng thuật toán / bước đột phá

**Tham lam theo vị trí.** Gọi `run` là số bản liên tiếp của ký tự cuối cùng đã đặt. Ở mỗi bước:
- **nếu** ký tự lớn nhất còn lại `c` khác ký tự cuối, hoặc `run < limit` **thì** đặt `c`;
- **ngược lại** (đã đủ `limit` bản `c`) đặt **một** bản của ký tự lớn nhất **nhỏ hơn** `c` còn lại; **nếu** không có **thì** dừng.

**Theo khối (nhanh hơn):** với ký tự lớn nhất `c` còn `cnt[c]`: đặt `t = min(cnt[c], limit)` bản. Nếu `c` vẫn còn: tìm ký tự `d < c` lớn nhất còn lại, đặt **một** `d`, quay lại `c`; nếu không có `d` thì dừng. Nếu `c` hết: chuyển sang ký tự lớn tiếp theo.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tham lam từng ký tự (mảng đếm, quét 26 ô mỗi bước)

In [7]:
def rls_char_by_char(s, k):
    cnt = [0] * 26
    for ch in s:
        cnt[ord(ch) - 97] += 1
    out = []
    last, run = -1, 0
    while True:
        placed = False
        for c in range(25, -1, -1):
            if cnt[c] == 0:
                continue
            if c == last and run == k:
                continue                         # đã đủ giới hạn: thử ký tự nhỏ hơn
            out.append(chr(97 + c)); cnt[c] -= 1
            run = run + 1 if c == last else 1
            last = c; placed = True
            break
        if not placed:
            return "".join(out)

for s, k, exp in examples:
    print(repr(rls_char_by_char(s, k)), "(kỳ vọng", repr(exp), ")")

'zzcccac' (kỳ vọng 'zzcccac' )
'bbabaa' (kỳ vọng 'bbabaa' )


**Độ phức tạp:** `O(n · 26)` — đủ nhanh (`2.6·10⁶` bước) nhưng mỗi ký tự đều quét lại từ `z`.

### 7.2 Cách 3 — Tham lam theo khối

In [8]:
def rls_blocks(s, k):
    cnt = [0] * 26
    for ch in s:
        cnt[ord(ch) - 97] += 1
    out = []
    c = 25
    while c >= 0:
        if cnt[c] == 0:
            c -= 1; continue
        t = min(cnt[c], k)
        out.append(chr(97 + c) * t); cnt[c] -= t
        if cnt[c] == 0:
            c -= 1; continue
        d = c - 1                                # tìm ký tự nhỏ hơn để ngắt
        while d >= 0 and cnt[d] == 0:
            d -= 1
        if d < 0:
            break
        out.append(chr(97 + d)); cnt[d] -= 1     # chỉ MỘT ký tự ngắt
    return "".join(out)

for s, k, exp in examples:
    print(repr(rls_blocks(s, k)), "(kỳ vọng", repr(exp), ")")

'zzcccac' (kỳ vọng 'zzcccac' )
'bbabaa' (kỳ vọng 'bbabaa' )


**Độ phức tạp:** mỗi vòng ngoài đặt ít nhất một ký tự; vòng tìm `d` quét lùi — xấu nhất `O(n · 26)`, thực tế nhanh vì đặt cả khối. Cách 5 cải tiến việc tìm `d` thành tổng `O(26)`.

### 7.3 Cách 4 — Max-heap các cặp `(ký tự, số lượng)`

In [9]:
def rls_heap(s, k):
    h = [(-ord(ch), c) for ch, c in Counter(s).items()]
    heapq.heapify(h)
    out = []
    while h:
        neg, c = heapq.heappop(h)
        t = min(c, k)
        out.append(chr(-neg) * t)
        if c > t:                                # còn dư: cần ký tự ngắt
            if not h:
                break
            neg2, c2 = heapq.heappop(h)
            out.append(chr(-neg2))
            if c2 > 1:
                heapq.heappush(h, (neg2, c2 - 1))
            heapq.heappush(h, (neg, c - t))
    return "".join(out)

for s, k, exp in examples:
    print(repr(rls_heap(s, k)), "(kỳ vọng", repr(exp), ")")

'zzcccac' (kỳ vọng 'zzcccac' )
'bbabaa' (kỳ vọng 'bbabaa' )


**Độ phức tạp:** `O(n log 26)` (heap tối đa 26 phần tử). Ý tưởng giống cách 3 nhưng heap tự cho "ký tự lớn nhất còn lại" và "ký tự lớn thứ hai" để ngắt.

### 7.4 Cách 5 — Hai con trỏ trên mảng đếm `O(n + 26)`

Con trỏ `i` chỉ ký tự lớn nhất còn lại, con trỏ `j < i` chỉ ứng viên ngắt. `j` chỉ **đi xuống** (ký tự đã hết không bao giờ có lại) ⇒ tổng số bước của `j` là `O(26)`.

In [10]:
def rls_two_pointers(s, k):
    cnt = [0] * 26
    for ch in s:
        cnt[ord(ch) - 97] += 1
    out = []
    i = 25
    j = 24
    while i >= 0:
        if cnt[i] == 0:
            i -= 1; continue
        t = min(cnt[i], k)
        out.append(chr(97 + i) * t); cnt[i] -= t
        if cnt[i] == 0:
            i -= 1; continue
        j = min(j, i - 1)
        while j >= 0 and cnt[j] == 0:
            j -= 1
        if j < 0:
            break
        out.append(chr(97 + j)); cnt[j] -= 1
    return "".join(out)

for s, k, exp in examples:
    print(repr(rls_two_pointers(s, k)), "(kỳ vọng", repr(exp), ")")

'zzcccac' (kỳ vọng 'zzcccac' )
'bbabaa' (kỳ vọng 'bbabaa' )


**Vì sao `j` không cần quay lên?** Khi `i` giảm, mọi ký tự trong `(j, i)` đều đã hết (đó là lý do `j` đi xuống qua chúng) — trừ khi `j` chưa từng xuống tới đó; `j = min(j, i − 1)` xử lý trường hợp này. **Độ phức tạp:** `O(n + 26)` thời gian (xuất xâu `O(n)`), `O(26)` bộ nhớ.

## 8. TRACE từng bước

### 8.1 Trace cách 3 (theo khối) trên hai ví dụ

In [11]:
def trace_blocks(s, k):
    cnt = [0] * 26
    for ch in s: cnt[ord(ch) - 97] += 1
    out = []; c = 25
    print(f"  s={s!r}, limit={k}")
    while c >= 0:
        if cnt[c] == 0:
            c -= 1; continue
        t = min(cnt[c], k)
        out.append(chr(97 + c) * t); cnt[c] -= t
        left = {chr(97 + x): cnt[x] for x in range(26) if cnt[x]}
        print(f"    đặt {t} × '{chr(97 + c)}' -> {''.join(out)!r}, còn {left}")
        if cnt[c] == 0:
            c -= 1; continue
        d = c - 1
        while d >= 0 and cnt[d] == 0: d -= 1
        if d < 0:
            print(f"    '{chr(97 + c)}' còn {cnt[c]} nhưng không có ký tự nhỏ hơn để ngắt -> dừng"); break
        out.append(chr(97 + d)); cnt[d] -= 1
        print(f"    ngắt bằng một '{chr(97 + d)}' -> {''.join(out)!r}")
    print("  =>", repr("".join(out)))
    return "".join(out)

for s_, k_, _ in examples:
    trace_blocks(s_, k_)

  s='cczazcc', limit=3
    đặt 2 × 'z' -> 'zz', còn {'a': 1, 'c': 4}
    đặt 3 × 'c' -> 'zzccc', còn {'a': 1, 'c': 1}
    ngắt bằng một 'a' -> 'zzccca'
    đặt 1 × 'c' -> 'zzcccac', còn {}
  => 'zzcccac'
  s='aababab', limit=2
    đặt 2 × 'b' -> 'bb', còn {'a': 4, 'b': 1}
    ngắt bằng một 'a' -> 'bba'
    đặt 1 × 'b' -> 'bbab', còn {'a': 3}
    đặt 2 × 'a' -> 'bbabaa', còn {'a': 1}
    'a' còn 1 nhưng không có ký tự nhỏ hơn để ngắt -> dừng
  => 'bbabaa'


### 8.2 Trace cách 2 (từng ký tự) trên `"zzzzb"`, `limit = 1` (ví dụ tự dựng — đã đối chiếu vét cạn, đối chiếu lại đề gốc)

In [12]:
def trace_char(s, k):
    cnt = Counter(s); out = []; last = None; run = 0
    while True:
        cand = [c for c in sorted(cnt, reverse=True) if cnt[c] and not (c == last and run == k)]
        if not cand:
            print(f"    không còn ký tự hợp lệ (còn {dict((c, v) for c, v in cnt.items() if v)}) -> dừng"); break
        c = cand[0]; cnt[c] -= 1; run = run + 1 if c == last else 1; last = c; out.append(c)
        print(f"    đặt '{c}' (run={run}) -> {''.join(out)!r}")
    return "".join(out)

r = trace_char("zzzzb", 1)
print("  =>", repr(r), "| vét cạn:", repr(rls_brute("zzzzb", 1)))

    đặt 'z' (run=1) -> 'z'
    đặt 'b' (run=1) -> 'zb'
    đặt 'z' (run=1) -> 'zbz'
    không còn ký tự hợp lệ (còn {'z': 2}) -> dừng
  => 'zbz' | vét cạn: 'zbz'


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def repeatLimitedString(self, s: str, repeatLimit: int) -> str:
        cnt = [0] * 26
        for ch in s:
            cnt[ord(ch) - 97] += 1
        out = []
        i, j = 25, 24
        while i >= 0:
            if cnt[i] == 0:
                i -= 1
                continue
            t = min(cnt[i], repeatLimit)
            out.append(chr(97 + i) * t)          # đặt cả khối ký tự lớn nhất
            cnt[i] -= t
            if cnt[i] == 0:
                i -= 1
                continue
            j = min(j, i - 1)
            while j >= 0 and cnt[j] == 0:        # ký tự lớn nhất nhỏ hơn i còn lại
                j -= 1
            if j < 0:
                break
            out.append(chr(97 + j))              # chỉ một ký tự để ngắt
            cnt[j] -= 1
        return "".join(out)

for s_, k_, exp in examples:
    assert Solution().repeatLimitedString(s_, k_) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Tham lam theo vị trí (cách 2) là tối ưu.** Gọi `G` là xâu tham lam, `O` là một xâu hợp lệ bất kỳ. Giả sử `G` và `O` trùng nhau ở `p` ký tự đầu. Tại vị trí `p`:
- Ký tự `G[p]` là ký tự **lớn nhất** có thể đặt (còn hàng, không vượt giới hạn với tiền tố chung). Nếu `O[p]` tồn tại thì `O[p] ≤ G[p]` (vì `O[p]` cũng phải hợp lệ với cùng tiền tố). Nếu `O[p] < G[p]` thì `O < G`.
- Nếu `G` dừng ở `p` (không còn ký tự hợp lệ) thì `O` cũng không thể có ký tự ở `p` ⇒ `O` là tiền tố của… chính `G` ⇒ `O ≤ G`.
- Nếu `O` dừng ở `p` còn `G` tiếp tục ⇒ `O` là tiền tố thật của `G` ⇒ `O < G`.
⇒ `G ≥ O` với mọi `O`. ∎

**Theo khối (cách 3–5) cho cùng xâu với cách 2.** Khi ký tự lớn nhất là `c`, tham lam từng ký tự sẽ đặt `c` liên tục cho tới khi hết hoặc chạm giới hạn (`min(cnt, limit)` bản). Nếu còn `c`, bước tiếp theo bị cấm `c`, nên đặt ký tự lớn nhất `d < c` — **một** bản, vì ngay sau đó `c` lại hợp lệ và lớn hơn `d`. Nếu không có `d`, tham lam dừng.

**Hai con trỏ:** `j` luôn là ký tự lớn nhất `< i` còn hàng — vì các ký tự trong `(j, i)` đã hết và không bao giờ tăng lại.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)`, `σ = 26`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n! · n)` | lớn |
| 2. Từng ký tự | `O(n · σ)` | `O(σ)` + kết quả |
| 3. Theo khối | `O(n · σ)` xấu nhất, nhanh thực tế | `O(σ)` |
| 4. Max-heap | `O(n log σ)` | `O(σ)` |
| 5. Hai con trỏ | `O(n + σ)` | `O(σ)` |

In [14]:
rnd = random.Random(2182)
tests = {
    "ngẫu nhiên, limit 1": ("".join(rnd.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(10**5)), 1),
    "toàn 'z' + 1 'a', limit 2": ("z" * 99999 + "a", 2),
    "'y' và 'z' xen, limit 1": ("yz" * 50000, 1),
}
for label_, (s_, k_) in tests.items():
    row = []
    for name, fn in [("char", rls_char_by_char), ("blocks", rls_blocks), ("heap", rls_heap), ("two_ptr", rls_two_pointers), ("Solution", Solution().repeatLimitedString)]:
        t0 = time.perf_counter(); r = fn(s_, k_); row.append(f"{name}={(time.perf_counter() - t0) * 1000:.0f}ms")
    print(f"{label_:<26} (độ dài kết quả {len(r)}): " + ", ".join(row))

ngẫu nhiên, limit 1        (độ dài kết quả 99707): char=44ms, blocks=13ms, heap=21ms, two_ptr=13ms, Solution=13ms


toàn 'z' + 1 'a', limit 2  (độ dài kết quả 5): char=5ms, blocks=6ms, heap=3ms, two_ptr=5ms, Solution=5ms


'y' và 'z' xen, limit 1    (độ dài kết quả 100000): char=27ms, blocks=17ms, heap=21ms, two_ptr=18ms, Solution=15ms


## 12. Bộ test

In [15]:
ALL = {
    "char": rls_char_by_char,
    "blocks": rls_blocks,
    "heap": rls_heap,
    "two_ptr": rls_two_pointers,
    "Solution": lambda s, k: Solution().repeatLimitedString(s, k),
}

def check(s, k, exp=None, brute=True):
    ref = rls_brute(s, k) if brute else rls_char_by_char(s, k)
    assert is_valid(s, k, ref)
    for name, fn in ALL.items():
        got = fn(s, k)
        assert got == ref, (name, s, k, got, ref)
    if exp is not None:
        assert ref == exp, (s, k, ref, exp)
    return ref

for s_, k_, exp in examples:
    check(s_, k_, exp)
print("Ví dụ đề: OK")

check("a", 1, "a")
check("aaaa", 1, "a")                     # không có ký tự ngắt
check("aaaa", 4, "aaaa")
check("zzzzb", 1, "zbz")
check("zzzab", 2, "zzbza")
check("abc", 1, "cba")
print("Edge case (tự dựng, kỳ vọng đã đối chiếu vét cạn): OK")

rnd = random.Random(2021)
for _ in range(700):
    n = rnd.randint(1, 7)
    s_ = "".join(rnd.choice(rnd.choice(["ab", "abc", "az", "xyz"])) for _ in range(n))
    check(s_, rnd.randint(1, n))
print("700 xâu ngắn so với vét cạn: OK")

for _ in range(20):
    n = rnd.randint(1, 3000)
    s_ = "".join(rnd.choice("abcdefghijklmnopqrstuvwxyz"[: rnd.randint(1, 26)]) for _ in range(n))
    check(s_, rnd.randint(1, n), brute=False)
print("20 xâu dài (đối chiếu chéo + validator): OK")

Ví dụ đề: OK
Edge case (tự dựng, kỳ vọng đã đối chiếu vét cạn): OK


700 xâu ngắn so với vét cạn: OK
20 xâu dài (đối chiếu chéo + validator): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Ngắt bằng **nhiều** ký tự nhỏ (đặt cả khối ký tự nhỏ)

In [16]:
def bug_break_block(s, k):
    cnt = Counter(s); out = []; chars = sorted(cnt, reverse=True); i = 0
    while i < len(chars):
        c = chars[i]
        if cnt[c] == 0: i += 1; continue
        t = min(cnt[c], k); out.append(c * t); cnt[c] -= t
        if cnt[c] == 0: i += 1; continue
        d = next((x for x in chars[i + 1:] if cnt[x]), None)
        if d is None: break
        t2 = min(cnt[d], k); out.append(d * t2); cnt[d] -= t2      # SAI: ngắt bằng cả khối
    return "".join(out)

s_, k_ = "zzzzaaa", 2
print("sai :", repr(bug_break_block(s_, k_)), "| đúng:", repr(Solution().repeatLimitedString(s_, k_)))

sai : 'zzaazza' | đúng: 'zzazzaa'


**Vì sao sai:** sau `zz` chỉ cần **một** `a` để lại được đặt `z`. Đặt `aa` làm vị trí thứ 4 là `a` thay vì `z`. **Sửa:** ngắt bằng đúng một ký tự.

### Lỗi 2 — Dừng ngay khi ký tự lớn nhất chạm giới hạn (không ngắt)

In [17]:
def bug_no_break(s, k):
    cnt = Counter(s); out = []
    for c in sorted(cnt, reverse=True):
        t = min(cnt[c], k); out.append(c * t)
        if cnt[c] > k:
            break                                    # SAI: lẽ ra phải chèn ký tự nhỏ hơn rồi tiếp tục
    return "".join(out)

s_, k_ = "cczazcc", 3
print("sai :", repr(bug_no_break(s_, k_)), "| đúng:", repr(Solution().repeatLimitedString(s_, k_)))

sai : 'zzccc' | đúng: 'zzcccac'


**Vì sao sai:** bỏ mất phần đuôi `"ac"` — xâu đúng dài hơn và lớn hơn. **Sửa:** chèn ký tự ngắt, quay lại ký tự lớn.

### Lỗi 3 — Bỏ qua ký tự lớn khi còn dư, chuyển hẳn sang ký tự nhỏ

In [18]:
def bug_drop_rest(s, k):
    cnt = Counter(s); out = []
    for c in sorted(cnt, reverse=True):
        out.append(c * min(cnt[c], k))               # SAI: phần dư của c bị vứt bỏ
    return "".join(out)

s_, k_ = "zzzzb", 1
print("sai :", repr(bug_drop_rest(s_, k_)), "| đúng:", repr(Solution().repeatLimitedString(s_, k_)))

sai : 'zb' | đúng: 'zbz'


**Sửa:** sau khi ngắt, phải **quay lại** ký tự lớn còn dư (`zbz` chứ không phải `zb`).

### Lỗi 4 — Sắp xâu giảm dần rồi "cắt" mọi run dài (không chèn)

In [19]:
def bug_sort_and_trim(s, k):
    t = "".join(sorted(s, reverse=True)); out = []
    for ch in t:
        if len(out) >= k and all(x == ch for x in out[-k:]):
            continue                                 # SAI: chỉ bỏ ký tự thừa, không sắp lại
        out.append(ch)
    return "".join(out)

s_, k_ = "cczazcc", 3
print("sai :", repr(bug_sort_and_trim(s_, k_)), "| đúng:", repr(Solution().repeatLimitedString(s_, k_)))

sai : 'zzccca' | đúng: 'zzcccac'


**Vì sao sai:** ký tự `c` thứ tư bị bỏ thay vì được đặt **sau** một `a`. Xâu đúng `zzcccac` dài hơn và lớn hơn `zzccca`. **Sửa:** tham lam với ký tự ngắt.

### Lỗi 5 — Hai con trỏ nhưng đặt lại `j = i − 1` mỗi lần (vẫn đúng nhưng mất `O(n + 26)`)

In [20]:
def steps_two_pointers(s, k, reset):
    cnt = [0] * 26
    for ch in s: cnt[ord(ch) - 97] += 1
    i, j, steps = 25, 24, 0
    while i >= 0:
        if cnt[i] == 0: i -= 1; continue
        t = min(cnt[i], k); cnt[i] -= t
        if cnt[i] == 0: i -= 1; continue
        j = i - 1 if reset else min(j, i - 1)
        while j >= 0 and cnt[j] == 0:
            j -= 1; steps += 1
        if j < 0: break
        cnt[j] -= 1
    return steps

s_ = "z" * 50000 + "a" * 50000
print("số bước dò j — đặt lại mỗi lần:", steps_two_pointers(s_, 1, True), "| giữ j:", steps_two_pointers(s_, 1, False))

số bước dò j — đặt lại mỗi lần: 1199976 | giữ j: 24


**Vì sao là lỗi:** kết quả đúng nhưng mỗi lần ngắt lại quét từ `y` xuống `a` — `24 × 50 000` bước thay vì 24. Với bảng chữ cái lớn hơn (ví dụ Unicode) sẽ chậm rõ rệt. **Sửa:** `j = min(j, i − 1)` — `j` chỉ đi xuống.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Xâu lớn/nhỏ nhất theo từ điển" + ràng buộc cục bộ ⇒ **tham lam theo vị trí**, chứng minh bằng "vị trí khác nhau đầu tiên".
- Bảng chữ cái nhỏ ⇒ mảng đếm 26 ô thay cho sắp xếp.
- Khi bị chặn: chèn **một** phần tử nhỏ nhất đủ để mở khóa, rồi quay lại phần tử lớn.
- Con trỏ chỉ đi một chiều trên mảng đếm ⇒ tổng `O(σ)`.
- Validator: kiểm đa tập con + độ dài run tối đa.

## 15. Biến thể của bài

1. **LeetCode 767 — Reorganize String** (`limit = 1`, phải dùng **hết** ký tự, bất kỳ thứ tự) — heap.
2. **LeetCode 1405 — Longest Happy String** (`a, b, c`, không 3 ký tự liền nhau, dài nhất).
3. **LeetCode 984 — String Without AAA or BBB.**
4. **Xâu nhỏ nhất** theo từ điển với cùng ràng buộc — đối xứng (duyệt từ `a`), nhưng lưu ý xâu **ngắn hơn** lại nhỏ hơn ⇒ đáp án tầm thường là xâu rỗng/một ký tự — nên biến thể thường bắt buộc dùng hết ký tự (bài tập 5).
5. **Giới hạn khác nhau cho từng ký tự** `limit[c]`.

In [21]:
def reorganize_string(s):
    """LeetCode 767: sắp lại để không có hai ký tự giống nhau liền nhau, hoặc ''."""
    h = [(-c, ch) for ch, c in Counter(s).items()]
    heapq.heapify(h); out = []; prev = None
    while h:
        c, ch = heapq.heappop(h)
        out.append(ch)
        if prev:
            heapq.heappush(h, prev)
        prev = (c + 1, ch) if c + 1 < 0 else None
    return "".join(out) if len(out) == len(s) else ""

rnd_v = random.Random(15)
for _ in range(1000):
    t = "".join(rnd_v.choice("aabbc") for _ in range(rnd_v.randint(1, 9)))
    r = reorganize_string(t)
    possible = max(Counter(t).values()) <= (len(t) + 1) // 2
    assert (r != "") == possible and (not r or (sorted(r) == sorted(t) and max_run(r) == 1))
print("767 'aab' ->", repr(reorganize_string("aab")), "| 'aaab' ->", repr(reorganize_string("aaab")), "| kiểm 1000 xâu: OK")

767 'aab' -> 'aba' | 'aaab' -> '' | kiểm 1000 xâu: OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | mọi xâu | mũ | lớn | kiểm thử |
| 2. Từng ký tự | lớn nhất hợp lệ | `O(nσ)` | `O(σ)` | dễ chứng minh |
| 3. Theo khối | khối + 1 ký tự ngắt | `O(nσ)` xấu nhất | `O(σ)` | trực quan |
| 4. Max-heap | heap cặp | `O(n log σ)` | `O(σ)` | khi σ lớn |
| 5. Hai con trỏ | `i`, `j` đi xuống | `O(n + σ)` | `O(σ)` | **nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().repeatLimitedString("aaabbbccc", 1)`?

**Bài 2 (tính tay):** `s = "zzzzzyyx"`, `limit = 2`. Viết từng khối và ký tự ngắt.

**Bài 3 (điền TODO):** Hoàn thiện `chars_unused(s, k)` — trả về `Counter` các ký tự **không** được dùng trong đáp án.

**Bài 4 (tự viết lại):** Viết lại cách 4 (heap) nhưng heap chứa **ký tự** (một phần tử cho mỗi loại), số lượng lưu ở mảng đếm riêng.

**Bài 5 (mini challenge):** `smallest_using_all(s, k)`: xâu **nhỏ nhất** theo từ điển, phải dùng **hết** ký tự, không run nào dài hơn `k`; trả `""` nếu không thể. Gợi ý: tham lam theo vị trí với **kiểm tra khả thi** cho phần còn lại (phần còn lại khả thi ⇔ ký tự nhiều nhất `m` thỏa `m ≤ k · (phần còn lại − m + 1)` — tính cả ràng buộc với ký tự vừa đặt). Kiểm bằng vét cạn.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def chars_unused(s, k):
    # TODO: Counter(s) - Counter(đáp án)
    pass

# Bài 4 — TODO
def rls_heap_chars(s, k):
    # TODO: heap các -ord(ch) cho ký tự có cnt > 0
    pass

# Bài 5 — TODO
def smallest_using_all(s, k):
    # TODO: tham lam từ 'a', kiểm khả thi phần còn lại trước khi đặt
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Đặt `c`, phải ngắt bằng `b`.
- Mức 2: `cbcbcb…`.
- Mức 3: Hết `c` và `b`, còn `a`: đặt **một** `a` (giới hạn 1), không có gì ngắt ⇒ `"cbcbcba"`.

**Bài 2**
- Mức 1: `zz` + ngắt `y`.
- Mức 2: `zzyzz` + ngắt `y` + `z`.
- Mức 3: `z` vừa hết; ký tự lớn nhất còn lại là `x` ⇒ `"zzyzzyzx"`.

**Bài 3**
- Mức 1: Tính đáp án.
- Mức 2: `Counter` trừ nhau.
- Mức 3: Ví dụ 2 thừa một `a`.

**Bài 4**
- Mức 1: Heap chỉ chứa ký tự còn hàng.
- Mức 2: Ký tự lớn nhất: `-h[0]`; ký tự ngắt: phần tử lớn thứ hai — pop tạm rồi push lại.
- Mức 3: Đối chiếu `Solution`.

**Bài 5**
- Mức 1: Thử ký tự từ `a` lên, chọn ký tự đầu tiên mà sau khi đặt phần còn lại vẫn khả thi.
- Mức 2: Khả thi (bỏ qua ký tự trước) ⇔ `max_count ≤ k · (others + 1)`; cần tính thêm: nếu ký tự nhiều nhất trùng ký tự vừa đặt với run `r`, nó chỉ được đặt thêm `k − r` ở đầu.
- Mức 3: Dễ sai — viết vét cạn để kiểm; có thể kiểm khả thi bằng chính tham lam "lớn nhất trước" với giới hạn (cách 2) xem có dùng hết không.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1
r1 = Solution().repeatLimitedString("aaabbbccc", 1)
print("Bài 1:", r1); assert r1 == "cbcbcba" == rls_char_by_char("aaabbbccc", 1)

# Bài 2
print("Bài 2:"); r2 = trace_blocks("zzzzzyyx", 2); assert r2 == "zzyzzyzx"

# Bài 3
def chars_unused_sol(s, k):
    return Counter(s) - Counter(Solution().repeatLimitedString(s, k))

print("Bài 3:", chars_unused_sol("aababab", 2), chars_unused_sol("cczazcc", 3))
assert chars_unused_sol("aababab", 2) == Counter({"a": 1})

Bài 1: cbcbcba
Bài 2:
  s='zzzzzyyx', limit=2
    đặt 2 × 'z' -> 'zz', còn {'x': 1, 'y': 2, 'z': 3}
    ngắt bằng một 'y' -> 'zzy'
    đặt 2 × 'z' -> 'zzyzz', còn {'x': 1, 'y': 1, 'z': 1}
    ngắt bằng một 'y' -> 'zzyzzy'
    đặt 1 × 'z' -> 'zzyzzyz', còn {'x': 1}
    đặt 1 × 'x' -> 'zzyzzyzx', còn {}
  => 'zzyzzyzx'
Bài 3: Counter({'a': 1}) Counter()


In [24]:
# Bài 4
def rls_heap_chars_sol(s, k):
    cnt = Counter(s); h = [-ord(c) for c in cnt]; heapq.heapify(h); out = []
    while h:
        c = chr(-heapq.heappop(h))
        t = min(cnt[c], k); out.append(c * t); cnt[c] -= t
        if cnt[c]:
            if not h:
                break
            d = chr(-h[0]); out.append(d); cnt[d] -= 1
            if cnt[d] == 0:
                heapq.heappop(h)
            heapq.heappush(h, -ord(c))
    return "".join(out)

# Bài 5
def can_use_all(cnt, last, run, k):
    """Có xếp hết cnt được không, biết ký tự cuối là last với run lần liên tiếp? (thử tham lam: ký tự nhiều nhất trước)."""
    cnt = dict(cnt)
    total = sum(cnt.values())
    while total:
        opts = sorted((v, c) for c, v in cnt.items() if v and not (c == last and run == k))
        if not opts:
            return False
        v, c = opts[-1]
        cnt[c] -= 1; total -= 1
        run = run + 1 if c == last else 1; last = c
    return True

def smallest_using_all_sol(s, k):
    cnt = Counter(s); out = []; last, run = None, 0
    if not can_use_all(cnt, None, 0, k):
        return ""
    for _ in range(len(s)):
        for c in sorted(cnt):
            if cnt[c] == 0 or (c == last and run == k):
                continue
            cnt[c] -= 1
            nr = run + 1 if c == last else 1
            if can_use_all(cnt, c, nr, k):
                out.append(c); last, run = c, nr; break
            cnt[c] += 1
    return "".join(out)

def smallest_using_all_brute(s, k):
    best = None
    for p in set(permutations(s)):
        t = "".join(p)
        if max_run(t) <= k and (best is None or t < best):
            best = t
    return best or ""

rnd = random.Random(5)
for _ in range(400):
    t = "".join(rnd.choice("aabbcz") for _ in range(rnd.randint(1, 7)))
    k = rnd.randint(1, 3)
    assert rls_heap_chars_sol(t, k) == Solution().repeatLimitedString(t, k)
    assert smallest_using_all_sol(t, k) == smallest_using_all_brute(t, k), (t, k)
print("Bài 4, Bài 5 OK; ví dụ:", repr(smallest_using_all_sol("aaabbc", 1)), repr(smallest_using_all_sol("aaab", 1)), repr(smallest_using_all_sol("cczazcc", 2)))

Bài 4, Bài 5 OK; ví dụ: 'ababac' '' 'acczccz'


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (hai con trỏ trên mảng đếm):**

```python
i, j = 25, 24
while i >= 0:
    if cnt[i] == 0: i -= 1; continue
    t = min(cnt[i], k); out.append(chr(97+i) * t); cnt[i] -= t
    if cnt[i] == 0: i -= 1; continue
    j = min(j, i - 1)
    while j >= 0 and cnt[j] == 0: j -= 1
    if j < 0: break
    out.append(chr(97+j)); cnt[j] -= 1        # MỘT ký tự ngắt
```

**Checklist:**
- [ ] Ngắt bằng **đúng một** ký tự nhỏ hơn?
- [ ] Sau khi ngắt, **quay lại** ký tự lớn còn dư?
- [ ] Dừng khi không còn ký tự nhỏ hơn để ngắt?
- [ ] Không cần dùng hết ký tự (khác bài 767)?
- [ ] Đã kiểm bằng validator + vét cạn trên xâu ngắn?
- [ ] Chữ ký đúng: `def repeatLimitedString(self, s: str, repeatLimit: int) -> str`?